# EcoGuard-ML Core: 03 Feature Engineering Pipeline

This notebook implements the production-grade feature engineering pipeline for the **EcoGuard-ML Core** threat forecasting platform. We transform raw spatial, temporal, climatic, and species variables into model-ready numeric arrays.

### Key Feature Engineering Steps:
1. **Clean Dataset Ingestion:** Load raw telemetry logs and impute missing weather records using the seasonal median parameters derived in EDA.
2. **Cyclical Temporal Encoding:** Project cyclic temporal columns (`hour`, `month`) into sine and cosine coordinates to maintain continuity.
3. **Categorical One-Hot Encoding:** Convert categoricals (`season`, `species`) to binary flags while excluding unique identifiers (`event_id`, `zone_id`).
4. **Stratified Splitting:** Partition the dataset into Train (70%), Validation (15%), and Test (15%) splits, using stratified splits on `poaching_incident` to prevent class representation bias.
5. **Data-Leakage-Free Standard Scaling:** Fit a `StandardScaler` strictly on the training set and transform the validation and test sets.
6. **Model-Ready Export:** Save the feature splits to `data/features/` and output verification diagnostics.

## Section 1: Imports & Configuration

We configure scientific imports and set seeds to ensure identical data distributions across multiple preprocess cycles.

In [1]:
import pandas as pd
import numpy as np
import os
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set seeds for absolute reproducibility
np.random.seed(42)

print("Imports successfully loaded.")

Imports successfully loaded.


## Section 2: Loading & Imputation

We load the raw CSV dataset and resolve the 155 missing values in `rainfall` using seasonal medians.

In [2]:
def load_and_impute_dataset(csv_path: str) -> pd.DataFrame:
    """
    Loads the raw telemetry dataset and performs seasonal median imputation
    to resolve missing rainfall cells.
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Target dataset not found at: {csv_path}")
        
    df = pd.read_csv(csv_path)
    print(f"Loaded raw dataset shape: {df.shape}")
    print(f"Null values before cleaning: {df['rainfall'].isnull().sum()}")
    
    # Apply seasonal median imputation
    seasonal_medians = df.groupby('season')['rainfall'].transform('median')
    df['rainfall'] = df['rainfall'].fillna(seasonal_medians)
    
    print(f"Null values after cleaning: {df['rainfall'].isnull().sum()}")
    return df

# Execute load
raw_csv_path = "../data/raw/master_dataset.csv"
if not os.path.exists(raw_csv_path):
    raw_csv_path = "data/raw/master_dataset.csv"
    
df_clean = load_and_impute_dataset(raw_csv_path)

Loaded raw dataset shape: (10000, 19)
Null values before cleaning: 155
Null values after cleaning: 0


## Section 3: Cyclical Temporal Encoding

### Why Cyclical Encoding is Used:
Hours (`0-23`) and Months (`1-12`) represent cyclical features. Standard numeric representations suggest that `23:00` is far from `00:00`, and December (`12`) is far from January (`1`). In reality, they are adjacent. 

To capture this continuity, we map these variables onto a 2D unit circle using trigonometric transformations. This creates coordinates where `hour_sin` and `hour_cos` represent cyclical positions, preventing distance calculation errors in modeling algorithms.

In [3]:
def encode_cyclical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies sine and cosine projections to hour and month columns.
    Drops original columns to avoid multi-collinearity.
    """
    df_encoded = df.copy()
    
    # Hour cyclical encoding
    df_encoded['hour_sin'] = np.sin(2 * np.pi * df_encoded['hour'] / 24.0)
    df_encoded['hour_cos'] = np.cos(2 * np.pi * df_encoded['hour'] / 24.0)
    
    # Month cyclical encoding
    df_encoded['month_sin'] = np.sin(2 * np.pi * df_encoded['month'] / 12.0)
    df_encoded['month_cos'] = np.cos(2 * np.pi * df_encoded['month'] / 12.0)
    
    # Drop original columns
    df_encoded = df_encoded.drop(columns=['hour', 'month'])
    
    print("Cyclical temporal coordinates computed successfully.")
    return df_encoded

df_temporal = encode_cyclical_features(df_clean)

Cyclical temporal coordinates computed successfully.


## Section 4: Categorical Encoding

We convert categorical strings (`season`, `species`) into numeric binary flags using One-Hot Encoding. 

We explicitly **do NOT** encode:
* **`event_id`**: A unique string hash index that has no predictive value (encoding it would create 10,000 sparse columns, causing model weight bloating).
* **`zone_id`**: The grid block identifier. While zones represent risk, their spatial coordinates are already captured by physical features (`latitude`, `longitude`, `elevation`, `distances`, and `historical_incident_count`). Over-encoding zones directly leads to structural overfitting.

In [4]:
def encode_categorical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies One-Hot Encoding to categorical variables.
    Keeps event_id and zone_id as raw identifier columns.
    """
    cols_to_encode = ['season', 'species']
    df_encoded = pd.get_dummies(df, columns=cols_to_encode, dtype=int)
    
    print(f"Categorical encoding completed. Total columns: {len(df_encoded.columns)}")
    return df_encoded

df_processed = encode_categorical_features(df_temporal)

Categorical encoding completed. Total columns: 29


## Section 5: Stratified Train / Validation / Test Splitting

We split the dataset into:
* **70% Training:** Used to fit the scaling parameters and train models.
* **15% Validation:** Used to tune hyperparameters.
* **15% Test:** Used to evaluate final generalization performance.

Because our target variable `poaching_incident` is highly imbalanced (7.5% incident rate), we apply **stratified sampling** to maintain identical class ratios across all three splits.

In [5]:
def perform_stratified_split(df: pd.DataFrame, target_col: str) -> tuple:
    """
    Partitions the dataset into Train (70%), Validation (15%), and Test (15%)
    using stratified splits on the target column.
    """
    # First split: Train (70%) and Temp (30%)
    train, temp = train_test_split(
        df,
        test_size=0.30,
        random_state=42,
        stratify=df[target_col]
    )
    
    # Second split: Validation (15%) and Test (15%)
    val, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=42,
        stratify=temp[target_col]
    )
    
    return train, val, test

train_df, val_df, test_df = perform_stratified_split(df_processed, 'poaching_incident')

## Section 6: Data-Leakage-Free Feature Scaling

We apply standard Z-score scaling (StandardScaler) to environmental, spatial, and count features. 

**CRITICAL:** To prevent data leakage, we fit the `StandardScaler` **strictly** on the training split (`train_df`), and then use the resulting means/stds to transform `train_df`, `val_df`, and `test_df`. We do not scale coordinate outputs, identifiers, or one-hot categorical flags.

In [6]:
def scale_numerical_features(train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame) -> tuple:
    """
    Fits StandardScaler on train set and scales numerical features across all splits.
    """
    cols_to_scale = [
        'temperature', 'humidity', 'rainfall', 'distance_to_road',
        'distance_to_water', 'distance_to_ranger_station', 'elevation',
        'animal_density_score', 'acoustic_risk', 'historical_incident_count'
    ]
    
    train_scaled = train.copy()
    val_scaled = val.copy()
    test_scaled = test.copy()
    
    scaler = StandardScaler()
    
    # Fit scaler strictly on training data
    scaler.fit(train_scaled[cols_to_scale])
    
    # Serialize StandardScaler to models directory for production API deployment
    import joblib
    models_dir = "../models"
    if not os.path.exists(models_dir):
        models_dir = "models"
    os.makedirs(models_dir, exist_ok=True)
    joblib.dump(scaler, os.path.join(models_dir, "scaler.pkl"))
    print("StandardScaler successfully fitted and saved to models/scaler.pkl")
    
    # Transform all splits
    train_scaled[cols_to_scale] = scaler.transform(train_scaled[cols_to_scale])
    val_scaled[cols_to_scale] = scaler.transform(val_scaled[cols_to_scale])
    test_scaled[cols_to_scale] = scaler.transform(test_scaled[cols_to_scale])
    
    print("Standard scaling completed without data leakage.")
    return train_scaled, val_scaled, test_scaled

train_scaled_df, val_scaled_df, test_scaled_df = scale_numerical_features(train_df, val_df, test_df)


Standard scaling completed without data leakage.


## Section 7: Saving Model-Ready Splits

We export the processed sets as CSVs to the features folder.

In [7]:
features_dir = "../data/features"
if not os.path.exists(features_dir):
    features_dir = "data/features"  # fallback
    
os.makedirs(features_dir, exist_ok=True)

train_scaled_df.to_csv(os.path.join(features_dir, "train.csv"), index=False)
val_scaled_df.to_csv(os.path.join(features_dir, "validation.csv"), index=False)
test_scaled_df.to_csv(os.path.join(features_dir, "test.csv"), index=False)

print(f"All splits successfully written to: {features_dir}/")

All splits successfully written to: data/features/


## Section 8: Verification & Diagnostics

We print the final diagnostics to verify the shapes, class distributions, and feature counts of our processed dataset.

In [8]:
print("=== Feature Engineering Verification ===\n")
print(f"Train set shape: {train_scaled_df.shape}")
print(f"Validation set shape: {val_scaled_df.shape}")
print(f"Test set shape: {test_scaled_df.shape}\n")

print("=== Target Class Distribution ===")
for name, split in [("Train", train_scaled_df), ("Validation", val_scaled_df), ("Test", test_scaled_df)]:
    val_counts = split['poaching_incident'].value_counts()
    pcts = split['poaching_incident'].value_counts(normalize=True) * 100
    print(f"{name} split:")
    print(f"  Count: Safe (0) = {val_counts[0]}, Threat (1) = {val_counts[1]}")
    print(f"  Ratio: Safe (0) = {pcts[0]:.2f}%, Threat (1) = {pcts[1]:.2f}%")

# Feature count (excluding target 'poaching_incident' and identifier columns)
identifiers = ['event_id', 'zone_id']
modeling_features = [col for col in train_scaled_df.columns if col not in identifiers + ['poaching_incident']]
print(f"\nTotal modeling features: {len(modeling_features)}")
print(f"Null values remaining: {train_scaled_df.isnull().sum().sum() + val_scaled_df.isnull().sum().sum() + test_scaled_df.isnull().sum().sum()}")

=== Feature Engineering Verification ===

Train set shape: (7000, 29)
Validation set shape: (1500, 29)
Test set shape: (1500, 29)

=== Target Class Distribution ===
Train split:
  Count: Safe (0) = 6475, Threat (1) = 525
  Ratio: Safe (0) = 92.50%, Threat (1) = 7.50%
Validation split:
  Count: Safe (0) = 1387, Threat (1) = 113
  Ratio: Safe (0) = 92.47%, Threat (1) = 7.53%
Test split:
  Count: Safe (0) = 1388, Threat (1) = 112
  Ratio: Safe (0) = 92.53%, Threat (1) = 7.47%

Total modeling features: 26
Null values remaining: 0
